Encoding word positions

batch_size: In the context of training a Large Language Model, batch size refers to the exact number of independent text samples (or sequences) that the neural network processes simultaneously in a single training step

max_length(Sequence Length): This dictates how many individual tokens (words or subwords) make up a single input sequence

embedding_dimension: This is the size of the continuous vector used to represent each individual token. The embedding layer's weight matrix is specifically configured to output 256 dimensions, meaning it translates one discrete token ID into a list of 256 numbers

In [20]:
# Import the GPTDatasetV1 and create_dataloader_v1 from data_sampling.ipynb
import sys
sys.path.append('../CH2-Working_with_text')

import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + 1 + max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = shuffle, drop_last = drop_last, num_workers = num_workers)
    return dataloader

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

Step 1: Create dataloader

In [21]:
max_length = 4

dataloader = create_dataloader_v1(
raw_text, batch_size=8, max_length=max_length,
stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


Step 2: Initialize the standard token embedding layer

In [22]:
import torch

vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

Step 3: Generate the token embeddings

In [23]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


Step 4: Initilize the positional embedding layer

In [24]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

Step 5: Retrive positional embeddings

Instead of passing text token IDs into this new layer, you pass a sequence of position indices (0, 1, 2, 3) using torch.arange. This fetches the specific positional vector for each slot in your context window

In [25]:
pos_embeddings = pos_embedding_layer(torch.arange(context_length))

Step 6: Combine the embeddings

Finally, you add the two tensors together. Because of PyTorch's "broadcasting" feature, the positional tensor is automatically duplicated and added to each of the 8 separate text sequences in your token embedding tensor

In [27]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


The resulting input_embeddings tensor now holds the continuous vector representations of your words that also carry mathematical information about their exact sequential order. This combined tensor is the final input that is fed into the main layers of the Large Language Model.